In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu google-genai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.0 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(pdf_path)

Saving dpdp_act.pdf to dpdp_act.pdf
dpdp_act.pdf


In [3]:
from pypdf import PdfReader

def load_text(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() for page in reader.pages)

text = load_text(pdf_path)
print(len(text))

63725


In [4]:
import re, json

def split_sections(text, max_section=44):
    pat = re.compile(r"(?m)^\s*(\d{1,2})\.\s+(?=[A-Z(“\"])")
    starts, nxt = [], 1
    for m in pat.finditer(text):
        if int(m.group(1)) == nxt and nxt <= max_section:
            starts.append((m.start(), m.group(1)))
            nxt += 1
    chunks = []
    for i, (pos, sec) in enumerate(starts):
        end = starts[i + 1][0] if i + 1 < len(starts) else len(text)
        chunks.append({"section": sec, "text": text[pos:end].strip()})
    return chunks

chunks = split_sections(text)
print(len(chunks), "sections")   # should be 44

def subchunk(items, max_chars=1500, overlap=200):
    out = []
    for c in items:
        t = c["text"]
        if len(t) <= max_chars:
            out.append(c)
        else:
            for p in range(0, len(t), max_chars - overlap):
                out.append({"section": c["section"], "text": t[p:p + max_chars]})
    return out

sub_chunks = subchunk(chunks)
print(len(sub_chunks), "sub-chunks")   # should be about 69

44 sections
69 sub-chunks


In [5]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def build_index(items):
    vecs = embedder.encode([i["text"] for i in items], normalize_embeddings=True, show_progress_bar=False)
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(np.array(vecs, dtype="float32"))
    return index

def retrieve(query, index, items, k=3):
    q = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    scores, ids = index.search(np.array(q, dtype="float32"), k)
    return [(items[i], float(s)) for i, s in zip(ids[0], scores[0])]

# fixed-size chunks, labelled with the section each window starts in
starts = []
for c in chunks:
    pos = text.find(c["text"][:60])
    if pos != -1:
        starts.append((pos, c["section"]))
starts.sort()

def section_at(pos):
    sec = starts[0][1]
    for p, s in starts:
        if p <= pos:
            sec = s
    return sec

fixed_chunks = []
size, overlap = 800, 100
for pos in range(0, len(text), size - overlap):
    fixed_chunks.append({"section": section_at(pos), "text": text[pos:pos + size]})

fixed_index   = build_index(fixed_chunks)
section_index = build_index(chunks)
sub_index     = build_index(sub_chunks)
print("indexes built:", len(fixed_chunks), len(chunks), len(sub_chunks))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

indexes built: 92 44 69


In [19]:
eval_set = [
    ("On what grounds can personal data be processed?", "4"),
    ("What must a data fiduciary tell a person when asking for consent?", "5"),
    ("What makes consent valid, and can it be withdrawn later?", "6"),
    ("What is a Consent Manager and how are they registered?", "6"),
    ("Can data be processed without consent if the person voluntarily gave it?", "7"),
    ("Can the government process data without consent to provide subsidies or benefits?", "7"),
    ("Is consent needed to process data during a medical emergency?", "7"),
    ("What security safeguards must a fiduciary put in place to prevent breaches?", "8"),
    ("What must a fiduciary do when a personal data breach happens?", "8"),
    ("When must a fiduciary erase personal data?", "8"),
    ("Who is responsible when a data processor handles data on a fiduciary's behalf?", "8"),
    ("What is required before processing a child's personal data?", "9"),
    ("Is tracking or targeted advertising directed at children allowed?", "9"),
    ("What extra obligations apply to significant data fiduciaries?", "10"),
    ("Can a person ask for a summary of the personal data held about them?", "11"),
    ("Can a person get their data corrected or erased?", "12"),
    ("Where should a person complain first about how their data is handled?", "13"),
    ("Can someone nominate another person to exercise their rights if they die or become incapacitated?", "14"),
    ("What duties does a data principal have under the Act?", "15"),
    ("Can personal data be transferred outside India?", "16"),
    ("When are government agencies exempt from the Act, for example for national security?", "17"),
    ("Are research or statistical purposes exempt from the Act?", "17"),
    ("Can a fiduciary offer a voluntary undertaking to the Board?", "32"),
    ("What penalties can the Board impose for breaches?", "33"),
    ("Can the government block access to a fiduciary's services in the public interest?", "37"),
    ("Can civil courts hear disputes that the Board is empowered to decide?", "39"),
   ("Is there an organization for data protection in India?", "18"),
    ("Do Board members need knowledge of the digital world?", "19"),
    ("Can a person with a physical disability be disqualified for the Data Protection Board?", "21"),
    ("How is the salary of the Board members decided?", "20"),
    ("How much notice must a board member give to resign", "22"),
]
print(len(eval_set), "questions")

31 questions


In [20]:
def hit_at_k(index, items, k=3):
    hits = 0
    for q, sec in eval_set:
        got = [it["section"] for it, _ in retrieve(q, index, items, k)]
        hits += sec in got
    return hits / len(eval_set)

print("Fixed-size hit@3:        ", round(hit_at_k(fixed_index, fixed_chunks), 3))
print("Section-aware hit@3:     ", round(hit_at_k(section_index, chunks), 3))
print("Section + sub-chunks hit@3:", round(hit_at_k(sub_index, sub_chunks), 3))

for k in (1, 3, 5):
    print(f"sub-chunks k={k}:", round(hit_at_k(sub_index, sub_chunks, k), 3))

print("\nMisses (section + sub-chunks):")
for q, sec in eval_set:
    got = [it["section"] for it, _ in retrieve(q, sub_index, sub_chunks, 3)]
    if sec not in got:
        print(f"- expected {sec}, got {got}: {q}")

Fixed-size hit@3:         0.581
Section-aware hit@3:      0.839
Section + sub-chunks hit@3: 0.903
sub-chunks k=1: 0.871
sub-chunks k=3: 0.903
sub-chunks k=5: 0.903

Misses (section + sub-chunks):
- expected 5, got ['6', '11', '9']: What must a data fiduciary tell a person when asking for consent?
- expected 7, got ['6', '9', '6']: Is consent needed to process data during a medical emergency?
- expected 21, got ['14', '9', '2']: Can a person with a physical disability be disqualified for the Data Protection Board?


In [8]:
def subchunk_ctx(items, max_chars=1500, overlap=200):
    out = []
    for c in items:
        t = c["text"]
        if len(t) <= max_chars:
            out.append(c)
            continue
        head = f"Section {c['section']}: " + t[:120].replace("\n", " ") + " ... "
        for i, p in enumerate(range(0, len(t), max_chars - overlap)):
            piece = t[p:p + max_chars]
            out.append({"section": c["section"], "text": piece if i == 0 else head + piece})
    return out

ctx_chunks = subchunk_ctx(chunks)
ctx_index = build_index(ctx_chunks)
print("Section + sub-chunks + context header hit@3:", round(hit_at_k(ctx_index, ctx_chunks), 3))

Section + sub-chunks + context header hit@3: 0.923


In [9]:
import time
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY").strip())
MODELS = ["gemini-3.8-flash", "gemini-3.5-flash"]   # tried in order

def generate_with_retry(prompt, retries=4):
    for attempt in range(retries):
        for model in MODELS:
            try:
                return client.models.generate_content(model=model, contents=prompt).text
            except Exception as e:
                msg = str(e)
                if "503" in msg or "429" in msg or "UNAVAILABLE" in msg:
                    continue          # try the next model
                raise                 # real error: show it
        time.sleep(2 ** attempt)      # wait 1s, 2s, 4s, 8s, then try again
    return "The model is busy right now. Please try again in a minute."

section_text = {c["section"]: c["text"] for c in chunks}

def answer(question, k=3):
    hits = retrieve(question, sub_index, sub_chunks, k)
    secs = []
    for h, _ in hits:                      # unique sections, in rank order
        if h["section"] not in secs:
            secs.append(h["section"])
    context = "\n\n".join(f"[Section {s}]\n{section_text[s]}" for s in secs)
    prompt = f"""You are a legal assistant for the Digital Personal Data Protection Act, 2023.
Answer ONLY from the context below. Cite section numbers like (Section 6).
If the context does not contain the answer, say "This is not covered in the retrieved sections."

Context:
{context}

Question: {question}
Answer:"""
    return generate_with_retry(prompt), hits

resp, hits = answer("When can a data fiduciary process data without consent?")
print(resp)
print("Sections given to the LLM:", sorted({h["section"] for h, _ in hits}, key=int))

Based on the provided context, a Data Fiduciary may process personal data of a Data Principal without standard consent under "Certain legitimate uses" in the following situations:

1. **Voluntary Provision:** For the specified purpose for which the Data Principal has voluntarily provided her personal data and has not indicated that she does not consent to its use (Section 7(a)).
2. **State Subsidies and Services:** For the State or its instrumentalities to provide or issue a prescribed subsidy, benefit, service, certificate, licence, or permit, if the data is available in a digital or digitised database, register, or document maintained by the State and notified by the Central Government (Section 7(b)(ii)).
3. **State Functions and Sovereignty:** For the performance of any function under law by the State or its instrumentalities, or in the interest of the sovereignty, integrity of India, or security of the State (Section 7(c)).
4. **Legal Disclosure Obligations:** For fulfilling any ob

In [10]:
def find(word):
    print(word, "->", [c["section"] for c in chunks if word.lower() in c["text"].lower()])

for w in ["medical emergency", "notice", "Consent Manager", "voluntary undertaking",
          "penalty", "block", "civil court", "outside India", "Appellate Tribunal"]:
    find(w)

medical emergency -> ['7']
notice -> ['2', '5', '6', '22', '40', '44']
Consent Manager -> ['2', '6', '13', '27', '40']
voluntary undertaking -> ['32', '44']
penalty -> ['27', '33', '37', '42', '44']
block -> ['37']
civil court -> ['28', '30', '39']
outside India -> ['7', '16', '17']
Appellate Tribunal -> ['2', '29', '30', '44']


In [11]:
for q in ["What is the penalty for tax evasion?",
          "Who won the 2023 cricket world cup?"]:
    resp, hits = answer(q)
    print("Q:", q, "\n", resp, "\n")

Q: What is the penalty for tax evasion? 
 This is not covered in the retrieved sections. 

Q: Who won the 2023 cricket world cup? 
 This is not covered in the retrieved sections. 



In [14]:
for w in ["quorum", "meetings", "proceedings of the Board"]:
    print(w, "->", [c["section"] for c in chunks if w.lower() in c["text"].lower()])

quorum -> []
meetings -> ['23']
proceedings of the Board -> []


In [21]:
import gradio as gr

def ask(q):
    resp, hits = answer(q)
    secs = []
    for h, s in hits:
        if h["section"] not in secs:
            secs.append(h["section"])
    return resp, "Sections used: " + ", ".join(secs)

gr.Interface(
    fn=ask,
    inputs=gr.Textbox(label="Ask about the DPDP Act 2023", lines=2),
    outputs=[gr.Textbox(label="Answer", lines=12), gr.Textbox(label="Sources")],
    title="DPDP Act Q&A (RAG)",
    examples=[["When can a data fiduciary process data without consent?"],
              ["What is required before processing a child's personal data?"]],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de511e0dc9f93dafdc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
